In [19]:
import pandas as pd

In [2]:
df = pd.read_csv('../green cleaned.csv', dtype={'ncusip': 'string'})
df['ret_fwd_1'] = df.groupby('permno')['ret_excess'].shift(-1)

In [29]:
import pandas as pd
import statsmodels.formula.api as smf

# 1. Load Portfolio Returns (Updated freq='ME' to fix deprecation warning)
port_df = pd.read_csv('../Residual NW GPT 3.5.csv')
port_df['Date'] = pd.date_range(start='2021-10-31', end='2024-04-30', freq='ME')
port_df.set_index('Date', inplace=True)

# 2. Load Local Fama-French 3-Factor Data
data_f = pd.read_csv('../AI Portfolio Selection/F-F_Research_Data_Factors.csv', sep=',')
data_f['Date'] = pd.to_datetime(data_f['Date'], format="%Y%m")
data_f['Date'] = data_f['Date'] - pd.DateOffset(months=1)
data_f['Date'] = data_f['Date'] + pd.offsets.MonthEnd(0)
data_f.set_index('Date', inplace=True)
data_f = data_f / 100

# 3. Load Momentum (MOM) Factor
mom_df = pd.read_csv('F-F_Momentum_Factor.csv', sep=',')
mom_df['Date'] = pd.to_datetime(mom_df['Date'], format="%Y%m")
mom_df['Date'] = mom_df['Date'] - pd.DateOffset(months=1)
mom_df['Date'] = mom_df['Date'] + pd.offsets.MonthEnd(0)
mom_df.set_index('Date', inplace=True)
mom_df = mom_df / 100 

# 4. Calculate Market-Wide Sentiment Factor
# Make sure the path matches where your FinBERT signals file is saved
finbert_df = pd.read_csv('../examples/monthly_signals_decay.csv') 
finbert_df['Date'] = pd.to_datetime(finbert_df['year_month']) + pd.offsets.MonthEnd(0)

# Calculate the mean sentiment score for all stocks in each month
mkt_sentiment_df = finbert_df.groupby('Date')['weighted_sentiment_score'].mean().reset_index()
mkt_sentiment_df.set_index('Date', inplace=True)
mkt_sentiment_df.rename(columns={'weighted_sentiment_score': 'mkt_sentiment'}, inplace=True)

# 5. Merge All Data (Now including mkt_sentiment_df)
final_df = port_df.join(data_f, how='inner').join(mom_df, how='inner').join(mkt_sentiment_df, how='inner')

# Strip invisible spaces and clean column names
final_df.columns = final_df.columns.str.strip()
final_df.rename(columns={'Mkt-RF': 'mkt_rf', 'Mom': 'mom'}, inplace=True)

# seeing what removing 40% outlier does
#final_df = final_df[final_df['portfolio_return'] < 0.40]

# 6. Run the 5-Factor Alpha Regression
# Added cov_type='HC3' to calculate robust standard errors (adjusts for your 47% outlier)
model = smf.ols(
    formula="portfolio_return ~ mkt_rf + SMB + HML + mom", 
    data=final_df
).fit(cov_type='HC3')

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       portfolio_return   R-squared:                       0.356
Model:                            OLS   Adj. R-squared:                  0.256
Method:                 Least Squares   F-statistic:                     4.163
Date:                Thu, 15 Oct 2026   Prob (F-statistic):            0.00975
Time:                        10:29:27   Log-Likelihood:                 35.209
No. Observations:                  31   AIC:                            -60.42
Df Residuals:                      26   BIC:                            -53.25
Df Model:                           4                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0261      0.017      1.511      0.1

In [27]:
# Assuming your fitted regression model is stored in the variable 'model'
latex_code = model.summary().as_latex()

# Print the LaTeX code so you can copy and paste it into Overleaf
print(latex_code)

\begin{center}
\begin{tabular}{lclc}
\toprule
\textbf{Dep. Variable:}    & portfolio\_return & \textbf{  R-squared:         } &     0.356   \\
\textbf{Model:}            &        OLS        & \textbf{  Adj. R-squared:    } &     0.256   \\
\textbf{Method:}           &   Least Squares   & \textbf{  F-statistic:       } &     4.163   \\
\textbf{Date:}             &  Thu, 15 Oct 2026 & \textbf{  Prob (F-statistic):} &  0.00975    \\
\textbf{Time:}             &      10:25:21     & \textbf{  Log-Likelihood:    } &    35.209   \\
\textbf{No. Observations:} &           31      & \textbf{  AIC:               } &    -60.42   \\
\textbf{Df Residuals:}     &           26      & \textbf{  BIC:               } &    -53.25   \\
\textbf{Df Model:}         &            4      & \textbf{                     } &             \\
\textbf{Covariance Type:}  &        HC3        & \textbf{                     } &             \\
\bottomrule
\end{tabular}
\begin{tabular}{lcccccc}
                   & \textbf{c